In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('../ai_company_adoption.csv')

selected_cols = [
    'ai_adoption_rate', 
    'ai_maturity_score', 
    'ai_failure_rate', 
    'task_automation_rate',
    'ai_training_hours', 
    'ai_adoption_stage', 
    'industry'
]

df_subset = df[selected_cols].copy()

num_cols = ['ai_adoption_rate', 'ai_maturity_score', 'ai_failure_rate', 'task_automation_rate']
for col in num_cols:
    if col in df_subset.columns:
        df_subset[col] = df_subset[col].fillna(df_subset[col].median())

cat_cols = ['ai_adoption_stage', 'industry']
for col in cat_cols:
    if col in df_subset.columns:
        df_subset[col] = df_subset[col].fillna(df_subset[col].mode()[0])

# cleaning before lcs implementation 
cleaned_filename = 'cleaned_ai_adoption_qm.csv'
df_subset.to_csv(cleaned_filename, index=False)
print(f"Cleaned data successfully saved to '{cleaned_filename}'!")

Cleaned data successfully saved to 'cleaned_ai_adoption_qm.csv'!


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

df_clean = pd.read_csv('cleaned_ai_adoption_qm.csv')

# non numerical feature
df_clean = pd.get_dummies(df_clean, columns=['industry'], drop_first=True)

# Label encode the target variable (ai_adoption_stage / LCS)
label_encoder = LabelEncoder()
target_col = 'ai_adoption_stage'
df_clean[target_col] = label_encoder.fit_transform(df_clean[target_col])

# Define features (X) and target (y)
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize and train the Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9999

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       337
           1       1.00      1.00      1.00      1040
           2       1.00      1.00      1.00     15760
           3       1.00      1.00      1.00     12863

    accuracy                           1.00     30000
   macro avg       1.00      1.00      1.00     30000
weighted avg       1.00      1.00      1.00     30000



In [8]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False))

ai_adoption_rate          0.703256
ai_maturity_score         0.132809
ai_failure_rate           0.065671
ai_training_hours         0.049315
task_automation_rate      0.046241
industry_Technology       0.001353
industry_Finance          0.000306
industry_Manufacturing    0.000230
industry_Retail           0.000207
industry_Healthcare       0.000177
industry_Logistics        0.000163
industry_Education        0.000149
industry_Consulting       0.000123
dtype: float64
